# Marlek YuE Colab Backend

This notebook runs the official YuE lyrics-to-song pipeline behind a small FastAPI bridge for the local Marlek Studio UI.

Use **Runtime > Change runtime type > T4 GPU**. YuE is much heavier than ACE-Step; start with the default one-segment test before trying longer generations.

In [ ]:
!nvidia-smi
!python --version


## Install YuE and tunnel tools

First run can take a long time because model weights and codec assets are downloaded from GitHub/Hugging Face.

In [ ]:
from pathlib import Path

%cd /content
!apt-get -qq update
!apt-get -qq install -y git-lfs
!git lfs install

if not Path('/content/YuE').exists():
    !git clone --depth 1 https://github.com/multimodal-art-projection/YuE.git /content/YuE

%cd /content/YuE
!pip install -q -r requirements.txt fastapi uvicorn nest_asyncio pydantic requests python-multipart soundfile

# FlashAttention is fragile on free Colab/Python 3.12. The next patch switches YuE to PyTorch SDPA.
!pip install -q bitsandbytes accelerate

%cd /content/YuE/inference
if not Path('/content/YuE/inference/xcodec_mini_infer').exists():
    !git clone https://huggingface.co/m-a-p/xcodec_mini_infer /content/YuE/inference/xcodec_mini_infer

# T4 does not have native bfloat16 tensor cores. Patch official inference to fp16 + SDPA for free Colab.
infer_path = Path('/content/YuE/inference/infer.py')
infer_text = infer_path.read_text(encoding='utf-8')
infer_text = infer_text.replace('torch_dtype=torch.bfloat16,', 'torch_dtype=torch.float16,')
infer_text = infer_text.replace('attn_implementation="flash_attention_2",', 'attn_implementation="sdpa",')
infer_text = infer_text.replace('if torch.__version__ >= "2.0.0":\n    model = torch.compile(model)', 'if False and torch.__version__ >= "2.0.0":\n    model = torch.compile(model)')
infer_path.write_text(infer_text, encoding='utf-8')

if not Path('/usr/local/bin/cloudflared').exists():
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

!cloudflared --version


## Start Marlek YuE API

Copy the printed `https://....trycloudflare.com` URL into the local app, choose `YuE`, then press `Baglanti testi` and `YuE ile uret`.

In [ ]:
import asyncio
import os
import random
import re
import shutil
import subprocess
import threading
import time
import traceback
import uuid
from pathlib import Path
from typing import Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

INFER_DIR = Path('/content/YuE/inference')
PUBLIC_OUTPUT_DIR = Path('/content/marlek_yue_audio')
JOB_ROOT = Path('/content/marlek_yue_jobs')
PUBLIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JOB_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
PORT = 7861
jobs = {}
jobs_lock = threading.Lock()

class GenerateRequest(BaseModel):
    model: str = 'YuE'
    prompt: str = Field(default='', max_length=5000)
    genre: str = Field(default='', max_length=1600)
    lyrics: str = Field(default='[verse]\nBozkirin yelinde kaldim bir gece', max_length=5000)
    seed: Optional[int] = None
    run_n_segments: int = Field(default=1, ge=1, le=2)
    max_new_tokens: int = Field(default=1000, ge=512, le=2000)
    stage2_batch_size: int = Field(default=1, ge=1, le=2)
    repetition_penalty: float = Field(default=1.1, ge=1.0, le=1.5)
    audio_duration: int = Field(default=30, ge=10, le=120)
    stage1_model: str = 'm-a-p/YuE-s1-7B-anneal-en-cot'
    stage2_model: str = 'm-a-p/YuE-s2-1B-general'

def first_yue_segment(text):
    clean = str(text or '').strip()
    if not clean:
        return '[verse]\nBozkirin yelinde kaldim bir gece\nGonlum saz telinde inler sessizce'
    sections = re.findall(r'\[(\w+)\](.*?)(?=\[|\Z)', clean, re.S)
    if sections:
        name, body = sections[0]
        lines = [line.strip() for line in body.splitlines() if line.strip()]
        return f'[{name}]\n' + '\n'.join(lines[:8])
    lines = [line.strip() for line in clean.splitlines() if line.strip()]
    return '[verse]\n' + '\n'.join(lines[:8])

def compose_genre(req):
    text = req.genre or req.prompt
    defaults = [
        'Turkish folk', 'Anatolian bozlak', 'Central Anatolian long air',
        'male vocal', 'native Turkish pronunciation', 'baglama saz lead',
        'bozuk duzen karaduzen baglama drone', 'kanun qanun tremolo',
        'ney breathy reed flute', 'kaval mey', 'oud',
        'darbuka bendir davul spoons cymbals', 'emotional', 'clean mix'
    ]
    joined = ', '.join([text, ', '.join(defaults)]) if text else ', '.join(defaults)
    return re.sub(r'\s+', ' ', joined).strip()[:1400]

def newest_audio_file(root):
    candidates = []
    for pattern in ('*.wav', '*.mp3', '*.flac'):
        candidates.extend(Path(root).rglob(pattern))
    candidates = [path for path in candidates if path.is_file() and path.stat().st_size > 1000]
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

def update_job(job_id, **updates):
    with jobs_lock:
        current = jobs.setdefault(job_id, {})
        current.update(updates)
        current['updated_at'] = time.time()

def response_for_audio(path, req, status='ok'):
    return {
        'status': status,
        'model': 'YuE',
        'seed': req.seed,
        'seconds': req.audio_duration,
        'filename': path.name,
        'audio_url': f'/audio/{path.name}',
        'note': 'YuE one-segment Colab generation. Longer full-song runs need more VRAM/time.'
    }

def run_yue_generation(req):
    if DEVICE != 'cuda':
        raise RuntimeError('YuE needs a CUDA GPU. In Colab choose Runtime > Change runtime type > T4 GPU.')
    seed = int(req.seed or random.randint(1, 2_147_483_000))
    req.seed = seed
    job_dir = JOB_ROOT / f'yue_{seed}_{uuid.uuid4().hex[:8]}'
    out_dir = job_dir / 'output'
    out_dir.mkdir(parents=True, exist_ok=True)
    genre_path = job_dir / 'genre.txt'
    lyrics_path = job_dir / 'lyrics.txt'
    genre_path.write_text(compose_genre(req), encoding='utf-8')
    lyrics_path.write_text(first_yue_segment(req.lyrics), encoding='utf-8')
    cmd = [
        'python', 'infer.py',
        '--cuda_idx', '0',
        '--stage1_model', req.stage1_model,
        '--stage2_model', req.stage2_model,
        '--genre_txt', str(genre_path),
        '--lyrics_txt', str(lyrics_path),
        '--run_n_segments', str(req.run_n_segments),
        '--stage2_batch_size', str(req.stage2_batch_size),
        '--output_dir', str(out_dir),
        '--max_new_tokens', str(req.max_new_tokens),
        '--repetition_penalty', str(req.repetition_penalty),
        '--seed', str(seed),
        '--rescale',
    ]
    env = os.environ.copy()
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    started = time.time()
    proc = subprocess.run(cmd, cwd=str(INFER_DIR), env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=90 * 60)
    (job_dir / 'infer.log').write_text(proc.stdout, encoding='utf-8', errors='replace')
    if proc.returncode != 0:
        tail = proc.stdout[-4000:]
        raise RuntimeError('YuE inference failed. Log tail:\n' + tail)
    audio_path = newest_audio_file(out_dir)
    if not audio_path:
        raise RuntimeError('YuE finished but no audio file was found. See infer.log in Colab.')
    suffix = audio_path.suffix.lower() or '.wav'
    public_path = PUBLIC_OUTPUT_DIR / f'marlek_yue_{seed}_{uuid.uuid4().hex[:8]}{suffix}'
    shutil.copy2(audio_path, public_path)
    print(f'YuE finished in {time.time() - started:.1f}s: {public_path}')
    return public_path

app = FastAPI(title='Marlek YuE Colab API')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=False,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/health')
async def health():
    return {
        'ok': True,
        'model': 'YuE',
        'device': DEVICE,
        'gpu': GPU_NAME,
        'mode': 'one-segment T4-safe Colab test',
    }

@app.get('/audio/{filename}')
async def audio(filename: str):
    safe = Path(filename).name
    path = PUBLIC_OUTPUT_DIR / safe
    if not path.exists():
        return JSONResponse({'status': 'error', 'error': 'audio not found'}, status_code=404)
    media_type = 'audio/mpeg' if path.suffix.lower() == '.mp3' else 'audio/wav'
    return FileResponse(path, media_type=media_type, filename=path.name)

@app.post('/generate')
async def generate(req: GenerateRequest):
    path = await asyncio.to_thread(run_yue_generation, req)
    return response_for_audio(path, req)

def worker(job_id, req):
    try:
        update_job(job_id, status='running', message='YuE model loading and generating one segment')
        path = run_yue_generation(req)
        update_job(job_id, **response_for_audio(path, req, status='done'))
    except Exception as exc:
        update_job(job_id, status='error', error=str(exc), traceback=traceback.format_exc()[-4000:])

@app.post('/generate_async')
async def generate_async(req: GenerateRequest, request: Request):
    job_id = uuid.uuid4().hex
    update_job(job_id, status='queued', message='YuE job queued', model='YuE', seed=req.seed)
    threading.Thread(target=worker, args=(job_id, req), daemon=True).start()
    return {'status': 'queued', 'job_id': job_id, 'poll_url': f'/jobs/{job_id}', 'model': 'YuE'}

@app.get('/jobs/{job_id}')
async def get_job(job_id: str):
    with jobs_lock:
        job = dict(jobs.get(job_id) or {})
    if not job:
        return JSONResponse({'status': 'error', 'error': 'job not found'}, status_code=404)
    return job

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

threading.Thread(target=start_server, daemon=True).start()
time.sleep(3)

cloudflared = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

PUBLIC_URL = None
lines_seen = []
deadline = time.time() + 45
while time.time() < deadline:
    line = cloudflared.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line, end='')
    lines_seen.append(line)
    match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if match:
        PUBLIC_URL = match.group(0)
        break

def drain_cloudflared():
    for line in cloudflared.stdout:
        if 'trycloudflare.com' in line or 'error' in line.lower():
            print(line, end='')

threading.Thread(target=drain_cloudflared, daemon=True).start()

if not PUBLIC_URL:
    raise RuntimeError('Cloudflare tunnel URL not found. Last output:\n' + ''.join(lines_seen[-20:]))

print('\nMARLEK_YUE_PUBLIC_URL=' + PUBLIC_URL)
print('Health:', PUBLIC_URL + '/health')
print('Async API:', PUBLIC_URL + '/generate_async')


## Optional YuE quality test inside Colab

Run this after the API cell if you want a direct Colab playback test. It can easily take 10-30+ minutes on a free T4 and may still OOM.

In [ ]:
import requests
import time
from IPython.display import Audio, display

test_payload = {
    'model': 'YuE',
    'genre': 'Turkish folk, Anatolian bozlak, Central Anatolian long air, male vocal, native Turkish pronunciation, baglama saz, karaduzen baglama, kanun, ney, darbuka, emotional clean mix',
    'lyrics': '[verse]\nBozkırın yelinde kaldım bir gece\nGönlüm saz telinde inler sessizce\nKırşehir yolunda yâr diye diye\nDağlar cevap verdi içimden ince',
    'run_n_segments': 1,
    'max_new_tokens': 1000,
    'stage2_batch_size': 1,
    'seed': 42842,
    'audio_duration': 30,
}

response = requests.post(PUBLIC_URL + '/generate_async', json=test_payload, timeout=120)
response.raise_for_status()
job = response.json()
print(job)
job_url = PUBLIC_URL + job['poll_url']

while True:
    payload = requests.get(job_url, timeout=60).json()
    print(payload.get('status'), payload.get('message') or payload.get('error') or payload.get('filename'))
    if payload.get('status') in ('done', 'ok', 'completed'):
        audio_url = PUBLIC_URL + payload['audio_url']
        print(audio_url)
        display(Audio(audio_url))
        break
    if payload.get('status') in ('error', 'failed'):
        raise RuntimeError(payload.get('error'))
    time.sleep(10)
